# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# The metadata is an object, use .name and .description attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
# Optionally, examine full metadata
# print(json.dumps(meta.to_json(), indent=2))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields, referencing each by its @id
record_sets = dataset.metadata.record_sets
print("Available Record Sets and associated fields/columns:")
overview = {}
for rs in record_sets:
    print(f"Record Set: {rs.name} (@id: {rs.id_})")
    overview[rs.id_] = {'name': rs.name, 'fields': [], 'columns': []}
    # List fields
    for f in rs.fields:
        print(f"  Field: {f.name} (@id: {f.id_}, dtype: {f.data_type})")
        overview[rs.id_]['fields'].append({'name': f.name, 'id': f.id_, 'dtype': f.data_type})
    # List columns
    for c in rs.columns:
        print(f"  Column: {c.name} (@id: {c.id_}, dtype: {c.data_type})")
        overview[rs.id_]['columns'].append({'name': c.name, 'id': c.id_, 'dtype': c.data_type})
    print()
# Store for later use
overview

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record set @ids (from overview above, e.g. ['cr:recordSet1', ...])
# For this dataset, there is likely one main record set (for tabular data)
# We'll choose the first one for demonstration.
record_sets_ids = [rs.id_ for rs in dataset.metadata.record_sets]
print(f"All record set @ids: {record_sets_ids}")
dataframes = {}

# Load data for each record set
for record_set_id in record_sets_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Fields for record set {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For EDA, we choose the first available record set with records
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break
if main_record_set_id is None:
    raise Exception("No tabular data found to analyze.")

df = dataframes[main_record_set_id]
print(f"Using record set: {main_record_set_id}")

# Find a numeric field (@id) for analysis
numeric_field_id = None
numeric_fields = [c['id'] for c in overview[main_record_set_id]['fields'] + overview[main_record_set_id]['columns'] if c['dtype'] in ['Float', 'Integer', 'Number']]
print(f"Numeric fields: {numeric_fields}")
for nf in numeric_fields:
    if nf in df.columns:
        numeric_field_id = nf
        break
if numeric_field_id is None:
    raise Exception("No numeric field found for EDA.")

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a group field
group_field_id = None
for c in overview[main_record_set_id]['fields'] + overview[main_record_set_id]['columns']:
    if c['id'] != numeric_field_id and c['id'] in df.columns and pd.api.types.is_object_dtype(df[c['id']]):
        group_field_id = c['id']
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group field (if available)
if group_field_id:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load and explore a FAIR-compliant dataset described using the Croissant schema with the `mlcroissant` library. We inspected the record sets, loaded records into pandas DataFrames using entity `@id`s, performed simple filtering, normalization, and grouping by field, and visualized distributions. This approach enables transparent, reproducible data processing workflows leveraging standardized dataset metadata.